In [2]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from PIL import Image
from torchvision import transforms, models

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
classes = [
    "crazing",
    "inclusion",
    "patches",
    "pitted_surface",
    "rolled_in_scale",
    "scratches"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=None)

model.fc = nn.Linear(
    model.fc.in_features,
    len(classes)
)

MODEL_PATH = os.path.join(
    "..",
    "models",
    "best_resnet18_neu_det.pth"
)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

model = model.to(device)
model.eval()

print("Model loaded successfully!")
print("Device:", device)
print("Classes:", classes)

Model loaded successfully!
Device: cpu
Classes: ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled_in_scale', 'scratches']


In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Image transformation ready!")

Image transformation ready!


In [5]:
def predict_image(image_path):

    image = Image.open(image_path).convert("RGB")

    image_tensor = transform(image)
    image_tensor = image_tensor.unsqueeze(0)
    image_tensor = image_tensor.to(device)

    with torch.no_grad():

        outputs = model(image_tensor)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        confidence, predicted_index = torch.max(
            probabilities,
            dim=1
        )

    predicted_class = classes[predicted_index.item()]
    confidence_value = confidence.item()

    return predicted_class, confidence_value

In [7]:
import os

validation_path = os.path.join(
    "..",
    "archive",
    "NEU-DET",
    "validation",
    "images"
)

# Find the first available image automatically
test_image = None

for class_name in classes:
    class_path = os.path.join(
        validation_path,
        class_name
    )

    if os.path.exists(class_path):
        image_files = [
            f for f in os.listdir(class_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        if image_files:
            test_image = os.path.join(
                class_path,
                image_files[0]
            )
            break

if test_image is None:
    print("No test image found.")
else:
    print("Test image found:")
    print(test_image)

Test image found:
..\archive\NEU-DET\validation\images\crazing\crazing_241.jpg


In [8]:
predicted_class, confidence = predict_image(test_image)

print("=" * 60)
print("SINGLE IMAGE PREDICTION")
print("=" * 60)

print("Image     :", os.path.basename(test_image))
print("Prediction:", predicted_class)
print("Confidence:", f"{confidence * 100:.2f}%")

print("=" * 60)

SINGLE IMAGE PREDICTION
Image     : crazing_241.jpg
Prediction: crazing
Confidence: 99.99%


In [9]:
validation_path = os.path.join(
    "..",
    "archive",
    "NEU-DET",
    "validation",
    "images"
)

results = []

for class_name in classes:

    class_path = os.path.join(
        validation_path,
        class_name
    )

    if not os.path.exists(class_path):
        print("Folder not found:", class_path)
        continue

    image_files = [
        f for f in os.listdir(class_path)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ]

    for image_file in image_files:

        image_path = os.path.join(
            class_path,
            image_file
        )

        predicted_class, confidence = predict_image(
            image_path
        )

        results.append({
            "Image": image_file,
            "Actual Class": class_name,
            "Predicted Class": predicted_class,
            "Confidence": confidence
        })

print("Batch prediction completed!")
print("Total images:", len(results))

Folder not found: ..\archive\NEU-DET\validation\images\rolled_in_scale
Batch prediction completed!
Total images: 300


In [10]:
results_df = pd.DataFrame(results)

results_df.head(10)

,Image,Actual Class,Predicted Class,Confidence
0,crazing_241.jpg,crazing,crazing,0.999864
1,crazing_242.jpg,crazing,crazing,0.999946
2,crazing_243.jpg,crazing,crazing,0.999407
3,crazing_244.jpg,crazing,crazing,0.999911
4,crazing_245.jpg,crazing,crazing,0.999964
5,crazing_246.jpg,crazing,crazing,0.999816
6,crazing_247.jpg,crazing,crazing,0.999914
7,crazing_248.jpg,crazing,crazing,0.999857
8,crazing_249.jpg,crazing,crazing,0.999969
9,crazing_250.jpg,crazing,crazing,0.999962


In [11]:
results_df["Correct"] = (
    results_df["Actual Class"]
    == results_df["Predicted Class"]
)

results_df.head()

,Image,Actual Class,Predicted Class,Confidence,Correct
0,crazing_241.jpg,crazing,crazing,0.999864,True
1,crazing_242.jpg,crazing,crazing,0.999946,True
2,crazing_243.jpg,crazing,crazing,0.999407,True
3,crazing_244.jpg,crazing,crazing,0.999911,True
4,crazing_245.jpg,crazing,crazing,0.999964,True


In [12]:
batch_accuracy = results_df["Correct"].mean()

print("=" * 60)
print("DAY 14 - BATCH INFERENCE SUMMARY")
print("=" * 60)

print(f"Total Images : {len(results_df)}")
print(f"Correct      : {results_df['Correct'].sum()}")
print(
    f"Wrong        : {(~results_df['Correct']).sum()}"
)
print(
    f"Accuracy     : {batch_accuracy * 100:.2f}%"
)

print("=" * 60)

DAY 14 - BATCH INFERENCE SUMMARY
Total Images : 300
Correct      : 300
Wrong        : 0
Accuracy     : 100.00%


In [13]:
class_summary = (
    results_df
    .groupby("Actual Class")
    .agg(
        Total=("Correct", "count"),
        Correct=("Correct", "sum")
    )
)

class_summary["Accuracy"] = (
    class_summary["Correct"]
    / class_summary["Total"]
    * 100
)

class_summary

,Total,Correct,Accuracy
Actual Class,,,
crazing,60,60,100.0
inclusion,60,60,100.0
patches,60,60,100.0
pitted_surface,60,60,100.0
scratches,60,60,100.0


In [14]:
confidence_summary = (
    results_df
    .groupby("Actual Class")["Confidence"]
    .mean()
    .mul(100)
    .round(2)
)

print("AVERAGE CONFIDENCE BY CLASS")
print("=" * 50)

for class_name, confidence in confidence_summary.items():

    print(
        f"{class_name:20s}: {confidence:.2f}%"
    )

AVERAGE CONFIDENCE BY CLASS
crazing             : 99.98%
inclusion           : 98.67%
patches             : 99.90%
pitted_surface      : 99.74%
scratches           : 99.87%


In [15]:
history_path = os.path.join(
    "..",
    "prediction_history.csv"
)

results_df.to_csv(
    history_path,
    index=False
)

print("Prediction history saved!")
print("File:", history_path)

Prediction history saved!
File: ..\prediction_history.csv


In [16]:
print("=" * 65)
print("DAY 14 - BATCH INFERENCE COMPLETED")
print("=" * 65)

print(f"Total Images Tested : {len(results_df)}")
print(f"Correct Predictions : {results_df['Correct'].sum()}")
print(f"Wrong Predictions   : {(~results_df['Correct']).sum()}")
print(f"Batch Accuracy      : {batch_accuracy * 100:.2f}%")

print("\nClasses Tested:")
for class_name in classes:
    print(f" - {class_name}")

print("\nPrediction history saved to:")
print("prediction_history.csv")

print("=" * 65)

DAY 14 - BATCH INFERENCE COMPLETED
Total Images Tested : 300
Correct Predictions : 300
Wrong Predictions   : 0
Batch Accuracy      : 100.00%

Classes Tested:
 - crazing
 - inclusion
 - patches
 - pitted_surface
 - rolled_in_scale
 - scratches

Prediction history saved to:
prediction_history.csv
